In [7]:
import os
from dotenv import load_dotenv
import mlflow
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Carrega o .env
load_dotenv()

# FORÇA a conexão direto com o servidor do Docker (evita olhar para a pasta mlruns local)
mlflow.set_tracking_uri("http://localhost:5000")

# Define o experimento unificado
mlflow.set_experiment("Recomendador_RetailRocket")

# Define a URI para a pasta raiz, garantindo que o notebook enxergue a pasta data
notebook_dir = os.getcwd()
base_dir = (
    os.path.abspath(os.path.join(notebook_dir, ".."))
    if notebook_dir.endswith("notebooks")
    else notebook_dir
)

# 2. Apontar para o seu arquivo correto: features_ready.csv
DATA_PATH = os.path.join(base_dir, "data", "processed", "features_ready.csv")

# 3. Carregar o DataFrame
df = pd.read_csv(DATA_PATH)
df["target"] = (df["target"] > 1).astype(int)

# Mantendo o target como 'target' conforme o pipeline
X = df.drop(columns=["target"])
y = df["target"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"MLflow conectado em: {mlflow.get_tracking_uri()}")
print(f"Dados de treino: {X_train.shape} | Validação: {X_val.shape}")

MLflow conectado em: http://localhost:5000
Dados de treino: (555853, 2) | Validação: (138964, 2)


In [8]:
mlflow.set_experiment("Recomendador_RetailRocket")

with mlflow.start_run(run_name="baseline_linear_regression"):
    # Criar o escalador e aplicar apenas nas features numéricas
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)

    # Treinando o modelo de regressão
    lr_model = LinearRegression()
    lr_model.fit(X_train_scaled, y_train)

    # Predições contínuas
    y_pred = lr_model.predict(X_val_scaled)

    # Métricas de regressão
    mse = mean_squared_error(y_val, y_pred)
    mae = mean_absolute_error(y_val, y_pred)
    r2 = r2_score(y_val, y_pred)

    # Salvando no MLflow
    mlflow.log_param("model_type", "linear_regression")
    mlflow.log_metric("mse", mse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2_score", r2)

    print(
        f"Regressão Linear -> MSE: {mse:.4f} | MAE: {mae:.4f} | R²: {r2:.4f}"
    )

Regressão Linear -> MSE: 0.2208 | MAE: 0.4421 | R²: 0.0000
🏃 View run baseline_linear_regression at: http://localhost:5000/#/experiments/1/runs/e9a08cd3a01f402699d0a7ab4714c4e8
🧪 View experiment at: http://localhost:5000/#/experiments/1


In [9]:
with mlflow.start_run(run_name="baseline_random_forest_regressor"):
    
    # Usando os dados originais (Árvores não precisam de scaler)
    rf_model = RandomForestRegressor(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1)
    rf_model.fit(X_train, y_train)
    
    # Predições
    y_pred = rf_model.predict(X_val)
    
    # Métricas
    mse = mean_squared_error(y_val, y_pred)
    mae = mean_absolute_error(y_val, y_pred)
    r2 = r2_score(y_val, y_pred)
    
    # Salvando no MLflow
    mlflow.log_param("model_type", "random_forest_regressor")
    mlflow.log_metric("mse", mse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2_score", r2)
    
    print(f"Random Forest Regressor -> MSE: {mse:.4f} | MAE: {mae:.4f} | R²: {r2:.4f}")

Random Forest Regressor -> MSE: 0.2190 | MAE: 0.4386 | R²: 0.0085
🏃 View run baseline_random_forest_regressor at: http://localhost:5000/#/experiments/1/runs/8647f5bea4b147048eafd28b5c31d2ff
🧪 View experiment at: http://localhost:5000/#/experiments/1
